In [9]:
import pcp
import plotly.graph_objects as go
import torch

In [2]:
points = pcp.io.read_csv("../../data/stanford_bunny.csv")

In [3]:
fps_points = pcp.sampling.fps(points, 500)
rand_points = pcp.sampling.random(points, 500)

In [4]:
p = points.to_numpy()
f = fps_points.to_numpy()
r = rand_points.to_numpy()
fig = go.Figure([
    go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers", marker=dict(size=2)),
    go.Scatter3d(x=f[:, 0], y=f[:, 1], z=f[:, 2], mode="markers", marker=dict(size=5, color="red")),
    go.Scatter3d(x=r[:, 0], y=r[:, 1], z=r[:, 2], mode="markers", marker=dict(size=5, color="green")),
])
fig.show()

In [5]:
centroid = points[0]
knn_points = pcp.grouping.knn(points, centroid, 200)

In [6]:
k = knn_points.to_numpy()
fig = go.Figure([
    go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers", marker=dict(size=2)),
    go.Scatter3d(x=k[:, 0], y=k[:, 1], z=k[:, 2], mode="markers", marker=dict(size=5, color="red"))
])
fig.show()

In [7]:
centroids = pcp.sampling.fps(points, k=500)
groups = pcp.grouping.knn(points, centroids, k=32)
print("Centroids shape:", centroids.shape)  # torch.Size([500, 3])
print("Groups shape:", groups.shape)  

Centroids shape: torch.Size([500, 3])
Groups shape: torch.Size([500, 32, 3])


In [10]:
pts = groups.points.reshape(-1, 3).numpy()
# 2. Assign each group a distinct random color index
M, K = groups.shape[0], groups.shape[1]
group_colors = torch.randperm(M).repeat_interleave(K).numpy()
# 3. Plot in a single fast trace
fig = go.Figure(
    go.Scatter3d(
        x=pts[:, 0],
        y=pts[:, 1],
        z=pts[:, 2],
        mode="markers",
        marker=dict(
            size=2.5,
            color=group_colors,
            colorscale="Turbo",
            opacity=0.9
        )
    )
)
fig.update_layout(
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, b=0, t=0)
)
fig.show()